## sklearn и линейная регрессия

Практический ноутбук для самостоятельной работы

На этом занятии мы:
* Узнаем о том, что такое переобучение, что такое линейная регрессия и как не допускать типичных ошибок
* Разберемся с основами библиотеки sklearn
* Узнаем о хороших практиках предобработки данных для линейных моделей


**Задание 1:** Импортируйте необходимые библиотеки для работы с данными и визуализацией.

Нам понадобятся:
- matplotlib.pyplot (как plt)
- seaborn (как sns)  
- pandas (как pd)
- numpy (как np)
- warnings (для отключения предупреждений)


In [ ]:
# Ваш код здесь
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline


## Часть 0. Введение в линейные модели

Линейная регрессия — это модель следующего вида:
$$a(x) = \langle w, x \rangle + w_0 = \sum_{j=1}^{d} w_j x_j + w_0$$

где $w_j$ — веса модели, которые мы подбираем в процессе обучения.

Модель обучается путем минимизации функции ошибки (например, MSE):
$$Q(a, X^\ell) = \frac{1}{\ell}\sum_{i=1}^{\ell}(a(x_i) - y_i)^2 \to \min_w$$

**Ridge-регрессия** добавляет к функции потерь регуляризацию:
$$Q(a, X^\ell) = \frac{1}{\ell}\sum_{i=1}^{\ell}(a(x_i) - y_i)^2 + \alpha \|w\|_2^2 \to \min_w$$

Регуляризация помогает бороться с переобучением!


**Задание 2:** Изучите пример переобучения. Запустите код ниже и объясните почему:
- Модель degree=1 недообучилась
- Модель degree=15 переобучилась
- Модель degree=4 работает лучше всего


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

np.random.seed(36)
x = np.linspace(0, 1, 100)
y = np.sin(1.5 * np.pi * x) + np.random.randn(100) * 0.3

# Разобьем данные на train и test
x_train, x_test = x[:50], x[50:]
y_train, y_test = y[:50], y[50:]

plt.figure(figsize=(15, 4))
for i, degree in enumerate([1, 4, 15]):
    plt.subplot(1, 3, i + 1)

    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(x_train.reshape(-1, 1), y_train)

    y_pred_train = model.predict(x_train.reshape(-1, 1))
    y_pred_test = model.predict(x_test.reshape(-1, 1))

    train_mse = np.mean((y_train - y_pred_train) ** 2)
    test_mse = np.mean((y_test - y_pred_test) ** 2)

    plt.scatter(x_train, y_train, label='train', alpha=0.5)
    plt.scatter(x_test, y_test, label='test', alpha=0.5)

    x_plot = np.linspace(0, 1, 200)
    plt.plot(x_plot, model.predict(x_plot.reshape(-1, 1)), 'r-', linewidth=2)

    plt.title(f'Degree={degree}\nTrain MSE: {train_mse:.3f}, Test MSE: {test_mse:.3f}')
    plt.legend()

plt.tight_layout()
plt.show()


**Ваш ответ:** (напишите объяснение здесь)



__Задание 2.1__. Попробуйте сделать Ridge и Lasso регуляризацию для максимальной степени полинома. Посмотрите на коэффициенты и визуализируйе предсказания. Что изменилось?

In [ ]:
# your code here

## Часть 1. Загружаем данные

Мы будем работать с данными о ценах на дома. Наша задача — предсказать цену дома (SalePrice) по его характеристикам.


In [ ]:
!wget -L -o 'train_sem2.csv' 'https://www.dropbox.com/s/6dxq90t0prn2vaw/_train_sem2.csv?dl=1' -s
# или !curl

**Задание 3:** Загрузите данные из файла `train_sem2.csv` и изучите их структуру.

Используйте:
- `pd.read_csv()` для загрузки
- `.head()` для просмотра первых строк
- `.shape` для размера
- `.columns` для списка колонок


In [ ]:
# Загрузите данные
data = # Ваш код здесь


In [ ]:
# Посмотрите на первые строки
# Ваш код здесь


In [ ]:
# Посмотрите на размер данных
# Ваш код здесь


In [ ]:
# Выведите список колонок
# Ваш код здесь


**Задание 4:** Подготовьте данные для обучения.

1. Удалите колонку "Id" (она не несет полезной информации)
2. Выделите целевую переменную "SalePrice" в переменную `y`
3. Остальные признаки сохраните в `X`
4. Разбейте данные на обучающую и тестовую выборки (test_size=0.2, random_state=10)

Подсказка: используйте `train_test_split` из `sklearn.model_selection`


In [ ]:
from sklearn.model_selection import train_test_split

# Удалите колонку Id
data = # Ваш код здесь

# Выделите целевую переменную
y = # Ваш код здесь

# Выделите признаки (все кроме SalePrice)
X = # Ваш код здесь

# Разбейте на train и test
X_train, X_test, y_train, y_test = # Ваш код здесь

print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")


**Задание 5:** Постройте гистограмму распределения целевой переменной `y_train`.

Что вы заметили в распределении? Есть ли выбросы?


In [ ]:
# Постройте гистограмму y_train
# Ваш код здесь


**Задание 6:** Найдите признаки, которые сильнее всего коррелируют с целевой переменной.

Для линейной регрессии полезно найти признаки с высокой корреляцией с целевой переменной.

1. Выберите только числовые признаки: `X_train.select_dtypes([np.number])`
2. Для каждого признака посчитайте корреляцию с y_train
3. Отсортируйте признаки по убыванию корреляции


In [ ]:
# Выберите числовые признаки
numeric_data = X_train.select_dtypes([np.number])
numeric_features = numeric_data.columns.tolist()

# Посчитайте корреляцию каждого признака с y_train
correlations = # Ваш код здесь (подсказка: используйте .corrwith() или цикл)

# Отсортируйте по убыванию абсолютного значения корреляции
# Ваш код здесь

print("Топ-10 признаков по корреляции:")
# Выведите результат


**Задание 7:** Постройте графики зависимости цены от трёх самых коррелирующих признаков.

Используйте `plt.scatter()` для построения графиков.


In [ ]:
# Выберите 3 лучших признака и постройте графики
fig, axs = plt.subplots(figsize=(16, 5), ncols=3)

# Ваш код здесь
# Для каждого из 3 признаков:
# - axs[i].scatter(X_train[feature], y_train)
# - axs[i].set_xlabel(feature)
# - axs[i].set_ylabel('SalePrice')

plt.tight_layout()
plt.show()


Помните, что корреляция $\neq$ зависимости. Модель должна быть наполнена экономическим смыслом, и найденные взаимосвязи вы должны уметь объяснять. Если такого не происходит, вы рискуете попасть в ситуацию ложной корреляции, как на рисунке ниже. Другие ложные корреляции можно увидеть на сайте [spurious correlations](https://tylervigen.com/spurious-correlations).

<a href="https://ibb.co/3yJmSHPs"><img src="https://i.ibb.co/4ZQR4x0f/corr-1.png" alt="corr-1" border="0"></a>



## Часть 2. Первая модель

Теперь обучим нашу первую модель! Будем использовать Ridge-регрессию из sklearn.

Основные шаги работы с моделями в sklearn:
1. Создать модель: `model = Ridge(alpha=1)`
2. Обучить на данных: `model.fit(X_train, y_train)`
3. Сделать предсказания: `model.predict(X_test)`
4. Оценить качество: например, посчитать RMSE


**Задание 8:** Обучите Ridge-регрессию на числовых признаках.

1. Создайте модель Ridge с alpha=1
2. Обучите её на числовых признаках обучающей выборки
3. Сделайте предсказания на тестовой выборке
4. Посчитайте RMSE (Root Mean Squared Error)

Формула RMSE: $\sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

# Создайте модель
model = # Ваш код здесь

# Заполните пропуски в числовых данных средними значениями
X_train_num = X_train[numeric_features].fillna(X_train[numeric_features].mean())
X_test_num = X_test[numeric_features].fillna(X_train[numeric_features].mean())

# Обучите модель
# Ваш код здесь

# Сделайте предсказания
y_pred = # Ваш код здесь

# Посчитайте RMSE
rmse = # Ваш код здесь (подсказка: mean_squared_error(y_test, y_pred, squared=False))

print(f"RMSE на тестовой выборке: {rmse:.2f}")


**Задание 9:** Оцените модель с помощью кросс-валидации.

Кросс-валидация — более надежный способ оценки качества модели. Данные разбиваются на K частей (фолдов), и модель обучается K раз, каждый раз используя одну часть для валидации.

<a href="https://ibb.co/SXphkTmB"><img src="https://i.ibb.co/nsSHX21j/1200px-Kfold-cv-diagram.png" alt="1200px-Kfold-cv-diagram" border="0"></a>

Используйте `cross_val_score` из sklearn. Обратите внимание: sklearn возвращает отрицательный MSE (neg_mean_squared_error), так что для получения RMSE нужно взять корень из минус этого значения.


In [ ]:
from sklearn.model_selection import cross_val_score

# Используйте cross_val_score с cv=5 фолдов
# scoring='neg_mean_squared_error'
cv_scores = # Ваш код здесь

# Преобразуйте в RMSE
cv_rmse = np.sqrt(-cv_scores)

print(f"RMSE по фолдам: {cv_rmse}")
print(f"Средний RMSE: {cv_rmse.mean():.2f} +/- {cv_rmse.std():.2f}")


**Задание 10:** Посчитайте baseline — предсказание константой.

Хорошая практика — сравнивать модель с простым baseline. Самый простой baseline — предсказывать среднее значение целевой переменной для всех объектов.


In [ ]:
# Посчитайте RMSE при предсказании средним значением
best_constant = # Ваш код здесь (среднее y_train)

baseline_rmse = # Ваш код здесь (RMSE если предсказывать best_constant для всех)

print(f"RMSE baseline (среднее): {baseline_rmse:.2f}")
print(f"Наша модель лучше baseline на: {baseline_rmse - rmse:.2f}")


## Масштабирование признаков

Для линейных моделей важно масштабировать признаки! Это делает регуляризацию более справедливой (все признаки в одном масштабе) и ускоряет сходимость.

**StandardScaler** преобразует признаки так, чтобы среднее было 0, а стандартное отклонение — 1:
$$x_{scaled} = \frac{x - \mu}{\sigma}$$


**Задание 11:** Примените StandardScaler к данным и обучите модель заново.

1. Создайте StandardScaler
2. Обучите его на обучающей выборке (fit)
3. Преобразуйте обе выборки (transform)
4. Обучите модель на масштабированных данных
5. Сравните RMSE с предыдущим результатом


In [ ]:
from sklearn.preprocessing import StandardScaler

# Создайте scaler
scaler = # Ваш код здесь

# Обучите на train и преобразуйте
X_train_scaled = # Ваш код здесь

# Преобразуйте test (только transform, не fit!)
X_test_scaled = # Ваш код здесь

# Обучите модель на масштабированных данных
model_scaled = Ridge(alpha=1)
# Ваш код здесь

# Посчитайте RMSE
y_pred_scaled = # Ваш код здесь
rmse_scaled = mean_squared_error(y_test, y_pred_scaled, squared=False)

print(f"RMSE без масштабирования: {rmse:.2f}")
print(f"RMSE с масштабированием: {rmse_scaled:.2f}")


**Задание 12:** Подберите оптимальный коэффициент регуляризации alpha.

Используйте GridSearchCV для перебора разных значений alpha и выбора лучшего по кросс-валидации.


In [ ]:
from sklearn.model_selection import GridSearchCV

# Создайте сетку значений alpha
alphas = np.logspace(-2, 3, 20)  # от 0.01 до 1000

# Создайте GridSearchCV
searcher = GridSearchCV(
    Ridge(),
    param_grid={'alpha': alphas},
    scoring='neg_mean_squared_error',
    cv=5
)

# Обучите на масштабированных данных
# Ваш код здесь

# Выведите лучшее значение alpha
print(f"Лучший alpha: {searcher.best_params_['alpha']:.4f}")
print(f"Лучший RMSE: {np.sqrt(-searcher.best_score_):.2f}")


## Использование Pipeline

Pipeline позволяет объединить несколько шагов обработки данных в одну цепочку. Это удобно и помогает избежать ошибок (например, "утечки данных" при масштабировании).


**Задание 13:** Создайте Pipeline из StandardScaler и Ridge.


In [ ]:
from sklearn.pipeline import Pipeline

# Создайте pipeline: сначала scaling, потом regression
simple_pipeline = Pipeline([
    # Ваш код здесь
    # ('scaling', ...),
    # ('regression', ...)
])

# Обучите pipeline
# Ваш код здесь

# Сделайте предсказания и посчитайте RMSE
y_pred_pipeline = simple_pipeline.predict(X_test_num)
rmse_pipeline = mean_squared_error(y_test, y_pred_pipeline, squared=False)
print(f"RMSE с Pipeline: {rmse_pipeline:.2f}")


## Часть 3. Работаем с категориальными признаками

До сих пор мы использовали только числовые признаки. Но в данных есть и категориальные! Например, тип района, стиль дома и т.д.

Линейная модель не может работать напрямую со строками. Нужно закодировать категории числами.

**One-Hot Encoding** — самый популярный способ:
- Создаём отдельный бинарный признак для каждой категории
- Если объект принадлежит категории — ставим 1, иначе 0


**Задание 14:** Найдите категориальные признаки в данных.


In [ ]:
# Найдите категориальные признаки (тип object)
categorical = # Ваш код здесь (подсказка: X_train.dtypes == "object")

print(f"Количество категориальных признаков: {len(categorical)}")
print(f"Примеры: {categorical[:5]}")


In [ ]:
# Заполните пропуски в категориальных признаках значением "NA"
X_train[categorical] = X_train[categorical].fillna("NA")
X_test[categorical] = X_test[categorical].fillna("NA")

# Посмотрите на примеры
X_train[categorical].sample(5)


**Задание 15:** Используйте ColumnTransformer для обработки разных типов признаков.

ColumnTransformer позволяет применять разные преобразования к разным колонкам:
- OneHotEncoder для категориальных
- StandardScaler для числовых


In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Создайте ColumnTransformer
column_transformer = ColumnTransformer([
    ('ohe', OneHotEncoder(handle_unknown="ignore"), categorical),
    ('scaling', StandardScaler(), numeric_features)
])

# Создайте pipeline с transformer и моделью
pipeline = Pipeline([
    ('preprocessing', column_transformer),
    ('regression', Ridge(alpha=1))
])

# Обучите и оцените
# Ваш код здесь


**Вопрос:** Почему мы не масштабируем One-Hot закодированные признаки?

**Ваш ответ:** (напишите здесь)


## Ridge vs Lasso

**Lasso** использует L1-регуляризацию вместо L2:
$$Q = \frac{1}{n}\sum(y_i - \hat{y}_i)^2 + \alpha \sum|w_j|$$

Главное отличие: Lasso может делать веса точно равными нулю, тем самым "выбрасывая" ненужные признаки.


**Задание 16:** Сравните Ridge и Lasso — сколько нулевых весов у каждой модели?


In [ ]:
from sklearn.linear_model import Lasso

# Создайте и обучите Ridge pipeline
ridge_pipeline = Pipeline([
    ('preprocessing', column_transformer),
    ('regression', Ridge(alpha=1))
])
ridge_pipeline.fit(X_train, y_train)

# Создайте и обучите Lasso pipeline
lasso_pipeline = Pipeline([
    ('preprocessing', column_transformer),
    ('regression', Lasso(alpha=1))
])
lasso_pipeline.fit(X_train, y_train)

# Посчитайте количество нулевых весов
ridge_zeros = np.sum(ridge_pipeline.steps[-1][-1].coef_ == 0)
lasso_zeros = np.sum(lasso_pipeline.steps[-1][-1].coef_ == 0)

print(f"Нулевых весов в Ridge: {ridge_zeros}")
print(f"Нулевых весов в Lasso: {lasso_zeros}")


**Задание 17:** Подберите оптимальный alpha для Lasso.


In [ ]:
# Создайте pipeline для Lasso
lasso_pipeline = Pipeline([
    ('preprocessing', column_transformer),
    ('regression', Lasso(max_iter=10000))
])

# Используйте GridSearchCV для подбора alpha
alphas = np.logspace(-2, 4, 20)
searcher = GridSearchCV(
    lasso_pipeline,
    param_grid={'regression__alpha': alphas},  # обратите внимание на синтаксис!
    scoring='neg_mean_squared_error',
    cv=5
)

# Обучите
# Ваш код здесь

print(f"Лучший alpha: {searcher.best_params_['regression__alpha']:.4f}")
print(f"Лучший RMSE: {np.sqrt(-searcher.best_score_):.2f}")


## Анализ остатков

Полезно смотреть на распределение ошибок модели. Это может помочь найти проблемные примеры.


**Задание 18:** Постройте гистограмму квадратов ошибок модели.


In [ ]:
# Обучите лучшую модель
best_pipeline = searcher.best_estimator_
y_pred_train = best_pipeline.predict(X_train)

# Посчитайте квадраты ошибок
errors_squared = (y_train - y_pred_train) ** 2

# Постройте гистограмму
# Ваш код здесь


**Задание 19:** Удалите выбросы и обучите модель заново.

Выбросами будем считать примеры с ошибкой больше 95-го перцентиля.


## Часть 4. Подготовка данных для линейных моделей

Есть важное понятие — *спрямляющее пространство*. Под ним понимается такое признаковое пространство для наших объектов, в котором линейная модель хорошо описывает данные.

Не существует общих рекомендаций о том, как найти спрямляющее пространство для произвольной выборки. Есть лишь некоторые общие советы — например, если добавить в выборку полиномиальных признаков, то модель может начать работать лучше.


### Разбиение признакового пространства

У линейных моделей есть огромное преимущество: они имеют мало параметров, а поэтому их можно обучить даже на небольшой выборке.

Иногда можно улучшить качество путём разбиения признакового пространства на несколько областей и построения своей модели в каждой из них.


**Задание 20:** Обучите базовую модель для сравнения.


In [ ]:
column_transformer = ColumnTransformer([
    ('ohe', OneHotEncoder(handle_unknown="ignore"), categorical),
    ('scaling', StandardScaler(), numeric_features)
])

pipeline = Pipeline(steps=[
    ('ohe_and_scaling', column_transformer),
    ('regression', Ridge())
])

model = pipeline.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Test RMSE = %.4f" % mean_squared_error(y_test, y_pred, squared=False))


**Задание 21:** Постройте график зависимости цены от OverallQual (общее качество дома).


In [ ]:
# Постройте scatter plot: X_train.OverallQual vs y_train
# Ваш код здесь

plt.figure(figsize=(7, 7))
# Ваш код здесь
plt.xlabel('OverallQual')
plt.ylabel('SalePrice')
plt.show()


**Задание 22:** Разбейте данные по признаку OverallQual и обучите две модели.

Идея: разбить данные на две группы (например, OverallQual <= 5 и > 5) и обучить отдельную модель для каждой группы.


In [ ]:
# Выберите порог для разбиения
threshold = 5

# Создайте маски для двух групп
mask = (X_train.OverallQual <= threshold)

# Разбейте данные
X_train_1 = X_train[mask]
y_train_1 = y_train[mask]
X_train_2 = X_train[~mask]
y_train_2 = y_train[~mask]

print(f"Группа 1 (OverallQual <= {threshold}): {len(X_train_1)} объектов")
print(f"Группа 2 (OverallQual > {threshold}): {len(X_train_2)} объектов")


In [ ]:
# Обучите две модели - по одной для каждой группы
column_transformer1 = ColumnTransformer([
    ('ohe', OneHotEncoder(handle_unknown="ignore"), categorical),
    ('scaling', StandardScaler(), numeric_features)
])

pipeline1 = Pipeline(steps=[
    ('ohe_and_scaling', column_transformer1),
    ('regression', Ridge())
])

column_transformer2 = ColumnTransformer([
    ('ohe', OneHotEncoder(handle_unknown="ignore"), categorical),
    ('scaling', StandardScaler(), numeric_features)
])

pipeline2 = Pipeline(steps=[
    ('ohe_and_scaling', column_transformer2),
    ('regression', Ridge())
])

# Обучите модели
# Ваш код здесь
pipeline1.fit(X_train_1, y_train_1)
pipeline2.fit(X_train_2, y_train_2)

# Сделайте предсказания на тесте (используя соответствующую модель для каждого объекта)
mask_test = (X_test.OverallQual <= threshold)
y_pred_combined = np.zeros(len(y_test))
y_pred_combined[mask_test] = pipeline1.predict(X_test[mask_test])
y_pred_combined[~mask_test] = pipeline2.predict(X_test[~mask_test])

# Посчитайте RMSE
rmse_combined = mean_squared_error(y_test, y_pred_combined, squared=False)
print(f"Test RMSE с разбиением: {rmse_combined:.4f}")


### Бинаризация признаков

Мы выбираем $n$ порогов $t_1, \dots, t_n$ для признака $x_j$ и генерируем $n+1$ новых бинарных признаков:
- $[x_j \leq t_1]$
- $[t_1 < x_j \leq t_2]$
- ...
- $[x_j > t_n]$

Такое преобразование может помочь, если целевая переменная нелинейно зависит от признака.


**Задание 23:** Изучите эффект бинаризации на синтетическом примере.

Запустите код ниже и объясните, почему бинаризация помогла.


In [ ]:
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.linear_model import LinearRegression

x_plot = np.linspace(0, 1, 10000)

X = np.random.uniform(0, 1, size=30)
y = np.cos(1.5 * np.pi * X) + np.random.normal(scale=0.1, size=X.shape)

fig, axs = plt.subplots(figsize=(16, 4), ncols=2)

# Без бинаризации
regr = LinearRegression()
regr.fit(X[:, np.newaxis], y)
y_pred_regr = regr.predict(x_plot[:, np.newaxis])
axs[0].scatter(X[:, np.newaxis], y, label="Данные")
axs[0].plot(x_plot, y_pred_regr, label="Предсказания", color='red')
axs[0].legend()
axs[0].set_title("Линейная регрессия без бинаризации")
axs[0].set_xlabel("$x$")
axs[0].set_ylabel("$y$")

# С бинаризацией
discretizer = KBinsDiscretizer(n_bins=10, encode="onehot", strategy="uniform")
X_bins = discretizer.fit_transform(X[:, np.newaxis])
regr_bins = LinearRegression()
regr_bins.fit(X_bins, y)

x_plot_bins = discretizer.transform(x_plot[:, np.newaxis])
y_pred_bins = regr_bins.predict(x_plot_bins)

axs[1].scatter(X[:, np.newaxis], y, label="Данные")
axs[1].plot(x_plot, y_pred_bins, label="Предсказания", color='red')
axs[1].legend()
axs[1].set_title("Линейная регрессия с бинаризацией (10 бинов)")
axs[1].set_xlabel("$x$")
axs[1].set_ylabel("$y$")

plt.tight_layout()
plt.show()


**Ваше объяснение:** (напишите здесь почему бинаризация помогла)



### Преобразование целевой переменной

Иногда целевая переменная меняется экспоненциально по мере роста признаков. Учесть это можно с помощью логарифмирования.


**Задание 24:** Изучите эффект логарифмирования на синтетическом примере.


In [ ]:
X = np.random.exponential(1, size=30)
y = np.exp(X) + np.random.normal(scale=0.1, size=X.shape)

x_plot = np.linspace(np.min(X), np.max(X), 10000)

fig, axs = plt.subplots(figsize=(16, 4), ncols=2)

# Без логарифмирования
regr = LinearRegression()
regr.fit(X[:, np.newaxis], y)
y_pred_regr = regr.predict(x_plot[:, np.newaxis])
axs[0].scatter(X[:, np.newaxis], y, label="Данные")
axs[0].plot(x_plot, y_pred_regr, label="Предсказания", color='red')
axs[0].legend()
axs[0].set_title("Линейная регрессия без преобразования")
axs[0].set_xlabel("$x$")
axs[0].set_ylabel("$y$")

# С логарифмированием целевой переменной
y_log = np.log(y)
regr_log = LinearRegression()
regr_log.fit(X[:, np.newaxis], y_log)
y_pred_log = np.exp(regr_log.predict(x_plot[:, np.newaxis]))

axs[1].scatter(X[:, np.newaxis], y, label="Данные")
axs[1].plot(x_plot, y_pred_log, label="Предсказания", color='red')
axs[1].legend()
axs[1].set_title("Линейная регрессия с log(y)")
axs[1].set_xlabel("$x$")
axs[1].set_ylabel("$y$")

plt.tight_layout()
plt.show()


**Задание 25:** Примените QuantileTransformer к целевой переменной в наших данных.

QuantileTransformer преобразует распределение к нормальному, что часто помогает линейным моделям.


In [ ]:
from sklearn.preprocessing import QuantileTransformer
from sklearn.compose import TransformedTargetRegressor

# Посмотрим на распределение целевой переменной до и после преобразования
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(14, 5))

ax0.hist(y_train, bins=30, density=True, alpha=0.7)
ax0.set_xlabel('Цена')
ax0.set_ylabel('Плотность')
ax0.set_title('Исходное распределение цен')

# Преобразуем целевую переменную
qt = QuantileTransformer(n_quantiles=300, output_distribution='normal')
y_train_transformed = qt.fit_transform(y_train.values.reshape(-1, 1)).ravel()

ax1.hist(y_train_transformed, bins=30, density=True, alpha=0.7)
ax1.set_xlabel('Преобразованная цена')
ax1.set_ylabel('Плотность')
ax1.set_title('После QuantileTransformer')

plt.tight_layout()
plt.show()


**Задание 26:** Обучите модель с преобразованием целевой переменной и сравните результаты.

Используйте `TransformedTargetRegressor` из sklearn.


In [ ]:
# TransformedTargetRegressor автоматически преобразует y при обучении
# и делает обратное преобразование при предсказании

from sklearn.compose import TransformedTargetRegressor

# Создайте модель с преобразованием целевой переменной
# Ваш код здесь

# Подсказка:
# regr_transformed = TransformedTargetRegressor(
#     regressor=pipeline,  # ваша модель
#     transformer=QuantileTransformer(n_quantiles=300, output_distribution='normal')
# )

# Обучите и оцените
# Ваш код здесь

# Сравните RMSE с моделью без преобразования


## Часть 5. Эконометрический взгляд

Эконометрика — это раздел прикладной математики, который изучает взаимосвязи между данными. В то время как классический ML заточен на получение лучшей метрики, эконометрика изучает причинно-следственные связи и качественные выводы из данных.

В эконометрике принята своя терминология:
- $y$ — **зависимая переменная**
- $x_i$ — **объясняющая переменная** (регрессор)

Классическое уравнение линейной регрессии записывают в виде:
$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \ldots + \beta_k x_k + \varepsilon$$

где $\varepsilon$ — случайная ошибка.


### Свойства оценок

**Несмещённость** оценки означает, что математическое ожидание оценки равно истинному значению параметра:
$$\mathbb{E}[\hat{\beta}] = \beta$$

**Эффективность** — оценка с наименьшей дисперсией среди всех несмещённых оценок.

Теорема Гаусса-Маркова утверждает, что МНК-оценки (метод наименьших квадратов) являются наилучшими линейными несмещёнными оценками (BLUE — Best Linear Unbiased Estimator).


**Задание 27:** Изучите несмещённость МНК-оценки на симуляции.

Запустите код ниже. Мы многократно генерируем выборки и считаем оценки коэффициентов. Если оценка несмещённая, среднее оценок должно быть близко к истинному значению.


In [ ]:
# Иллюстрация идеи несмещённости
np.random.seed(42)

# Параметры "истинной" модели
beta_0_true = 1.0
beta_1_true = 2.0
sigma = 1.0          # стандартное отклонение шума

n = 50               # размер выборки в одном эксперименте
n_sims = 10_000      # число повторов эксперимента

# Фиксируем x
x = np.random.uniform(0, 1, size=n)
X = np.column_stack([np.ones(n), x])  # матрица регрессоров

beta_0_hats = np.empty(n_sims)
beta_1_hats = np.empty(n_sims)

for i in range(n_sims):
    # Генерируем случайный шум с нулевым средним
    eps = np.random.normal(0, sigma, size=n)
    y = beta_0_true + beta_1_true * x + eps

    # МНК-оценка
    beta_hat = np.linalg.lstsq(X, y, rcond=None)[0]
    beta_0_hats[i] = beta_hat[0]
    beta_1_hats[i] = beta_hat[1]

# Визуализация
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

axs[0].hist(beta_0_hats, bins=50, density=True, alpha=0.7)
axs[0].axvline(beta_0_true, color='red', linestyle='--', linewidth=2, label=f'Истинное β₀ = {beta_0_true}')
axs[0].axvline(beta_0_hats.mean(), color='green', linestyle='-', linewidth=2, label=f'Среднее оценок = {beta_0_hats.mean():.4f}')
axs[0].set_title('Распределение оценок β₀')
axs[0].legend()

axs[1].hist(beta_1_hats, bins=50, density=True, alpha=0.7)
axs[1].axvline(beta_1_true, color='red', linestyle='--', linewidth=2, label=f'Истинное β₁ = {beta_1_true}')
axs[1].axvline(beta_1_hats.mean(), color='green', linestyle='-', linewidth=2, label=f'Среднее оценок = {beta_1_hats.mean():.4f}')
axs[1].set_title('Распределение оценок β₁')
axs[1].legend()

plt.tight_layout()
plt.show()

print(f"Истинное β₀ = {beta_0_true}, среднее оценок = {beta_0_hats.mean():.4f}")
print(f"Истинное β₁ = {beta_1_true}, среднее оценок = {beta_1_hats.mean():.4f}")


**Задание 28:** Что произойдёт, если ошибки имеют ненулевое среднее?

Измените код выше: вместо `eps = np.random.normal(0, sigma, size=n)` используйте `eps = np.random.normal(0.5, sigma, size=n)`.

Будет ли оценка β₁ всё ещё несмещённой? А оценка β₀?


In [ ]:
# Ваш эксперимент здесь
# Скопируйте код выше и измените мат. ожидание ошибок на 0.5

# Ваш код здесь


**Ваш ответ:** (объясните результаты)



**Задание 29:** Сравните эффективность оценок.

Сравним две оценки β₁:
1. МНК по всей выборке (эффективная)
2. МНК только по половине выборки (тоже несмещённая, но менее эффективная)


In [ ]:
np.random.seed(42)

beta_0_true = 1.0
beta_1_true = 2.0
sigma = 1.0

n = 50
n_sims = 10_000

x = np.random.uniform(0, 1, size=n)
X_full = np.column_stack([np.ones(n), x])
X_half = np.column_stack([np.ones(n//2), x[:n//2]])

beta_1_full = np.empty(n_sims)
beta_1_half = np.empty(n_sims)

for i in range(n_sims):
    eps = np.random.normal(0, sigma, size=n)
    y = beta_0_true + beta_1_true * x + eps

    # Оценка по всей выборке
    beta_full = np.linalg.lstsq(X_full, y, rcond=None)[0]
    beta_1_full[i] = beta_full[1]

    # Оценка по половине выборки
    beta_half = np.linalg.lstsq(X_half, y[:n//2], rcond=None)[0]
    beta_1_half[i] = beta_half[1]

# Визуализация
plt.figure(figsize=(10, 5))
plt.hist(beta_1_full, bins=50, density=True, alpha=0.7, label=f'Вся выборка, std = {beta_1_full.std():.4f}')
plt.hist(beta_1_half, bins=50, density=True, alpha=0.7, label=f'Половина выборки, std = {beta_1_half.std():.4f}')
plt.axvline(beta_1_true, color='red', linestyle='--', linewidth=2, label=f'Истинное β₁ = {beta_1_true}')
plt.legend()
plt.title('Сравнение эффективности оценок')
plt.show()

print(f"Дисперсия оценки по всей выборке: {beta_1_full.var():.4f}")
print(f"Дисперсия оценки по половине: {beta_1_half.var():.4f}")


**Вопрос:** В эксперименте мы считали оценку только по половине выборки, что очевидно влечёт бо́льшую дисперсию оценки. Однако когда это может иметь смысл?


**Ваш ответ:** (подумайте о выбросах и гетерогенных данных)

